In [ ]:
!pip install lightkurve transitleastsquares openpyxl -q

import lightkurve as lk
import numpy
import pandas as pd
import matplotlib.pyplot as plt
import os
import requests
import io
from datetime import datetime
from astropy.io import fits
from astropy.stats import sigma_clip
from transitleastsquares import (
    transitleastsquares,
    cleaned_array,
    catalog_info,
    transit_mask
)
from google.colab import files

plt.rcParams["figure.dpi"] = 500
print("Everything Imported Successfully")

Everything Imported Successfully


In [ ]:
# Dispositions and what they mean:
#   KP = Known Planet       → skip
#   CP = Confirmed Planet   → skip
#   PC = Planet Candidate   → skip
#   EB = Eclipsing Binary   → skip
#   FP = False Positive     → do NOT skip, still worth studying

TOI_CATALOG_URL = (
    "https://exoplanetarchive.ipac.caltech.edu/cgi-bin/nstedAPI/"
    "nph-nstedAPI?table=toi&select=tid,toi,tfopwg_disp&format=csv"
)
SKIP_DISPOSITIONS = {'KP', 'CP', 'PC', 'EB'}

print("Downloading TOI catalog from NASA Exoplanet Archive...")
try:
    resp = requests.get(TOI_CATALOG_URL, timeout=60)
    resp.raise_for_status()
    toi_df = pd.read_csv(io.StringIO(resp.text), comment='#')
    toi_df.columns = toi_df.columns.str.strip()
    toi_df['tid'] = pd.to_numeric(toi_df['tid'], errors='coerce')
    toi_df = toi_df.dropna(subset=['tid'])
    toi_df['tid'] = toi_df['tid'].astype(int)
    toi_lookup = toi_df.groupby('tid')['tfopwg_disp'].apply(list).to_dict()
    print(f"✓ TOI catalog loaded: {len(toi_df)} entries, {len(toi_lookup)} unique TIC IDs")
    print(f"  Dispositions present: {toi_df['tfopwg_disp'].unique()}")
except Exception as e:
    print(f"✗ Failed: {e}\n  Proceeding without TOI check — all stars will be analyzed.")
    toi_lookup = {}

✓ TOI catalog loaded: 7931 entries, 7622 unique TIC IDs
  Dispositions present: ['FP' 'PC' 'APC' 'KP' 'FA' 'CP' nan]


In [ ]:
def check_toi_status(tic_id):
    """
    Returns (should_skip: bool, reason: str, dispositions: list)
    """
    tic_int = int(tic_id)
    if tic_int not in toi_lookup:
        return False, "Not in TOI catalog — proceed", []

    disps = toi_lookup[tic_int]
    disp_set = set(str(d).strip().upper() for d in disps if pd.notna(d))

    if disp_set & SKIP_DISPOSITIONS:
        matched = disp_set & SKIP_DISPOSITIONS
        labels = {'KP': 'Known Planet', 'CP': 'Confirmed Planet',
                  'PC': 'Planet Candidate', 'EB': 'Eclipsing Binary'}
        reasons = [labels.get(d, d) for d in matched]
        return True, ', '.join(reasons), disps

    return False, f"In TOI catalog as {disps} — proceeding (FP)", disps

In [ ]:
print("Upload your target CSV (e.g. all_targets_S082_v1.csv):")
uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]

targets_df = pd.read_csv(csv_filename, comment='#', header=None)
targets_df.columns = ['TICID', 'Camera', 'CCD', 'Tmag', 'RA', 'Dec']
targets_df = targets_df[targets_df['TICID'] != 'TICID']
targets_df['TICID'] = pd.to_numeric(targets_df['TICID'], errors='coerce')
targets_df['Tmag']  = pd.to_numeric(targets_df['Tmag'],  errors='coerce')  # ← add this
targets_df['RA']    = pd.to_numeric(targets_df['RA'],    errors='coerce')  # ← and this
targets_df['Dec']   = pd.to_numeric(targets_df['Dec'],   errors='coerce')  # ← and this
targets_df = targets_df.dropna(subset=['TICID'])
targets_df['TICID'] = targets_df['TICID'].astype(int)

print(f"✓ Loaded {len(targets_df)} targets from {csv_filename}")
print(f"  Tmag range: {targets_df['Tmag'].min():.1f} – {targets_df['Tmag'].max():.1f}")
print(targets_df.head(5).to_string(index=False))

Upload your target CSV (e.g. all_targets_S082_v1.csv):


Saving all_targets_S101_v1 (1).csv to all_targets_S101_v1 (1).csv
✓ Loaded 13000 targets from all_targets_S101_v1 (1).csv
  Tmag range: -1.0 – 20.9
  TICID Camera CCD  Tmag       RA      Dec
1628071      1   3  3.99 188.0144 -16.1964
1650025      1   3  9.40 188.2020 -16.5770
5630042      1   4 16.25 178.9281 -23.5659
5647251      1   4 11.60 178.9385 -18.9111
5648410      1   4  9.03 178.9977 -20.6372


In [ ]:
LOG_DIR  = 'tess_results'
LOG_CSV  = os.path.join(LOG_DIR, 'results_log.csv')
LOG_XLSX = os.path.join(LOG_DIR, 'results_log.xlsx')
BASE_URL = 'https://mast.stsci.edu/api/v0.1/Download/file?uri='

os.makedirs(LOG_DIR, exist_ok=True)


def append_log(row_dict):
    """Append one result row to both CSV and XLSX."""
    import csv
    row_df = pd.DataFrame([row_dict])
    write_header = not os.path.exists(LOG_CSV)
    row_df.to_csv(LOG_CSV, mode='a', header=write_header, index=False, quoting=csv.QUOTE_ALL)

    if os.path.exists(LOG_XLSX):
        combined = pd.concat([pd.read_excel(LOG_XLSX), row_df], ignore_index=True)
    else:
        combined = row_df
    combined.to_excel(LOG_XLSX, index=False)


def run_tls_on_sector(tic_id, uri, sector_label, ab):
    """Download one sector FITS file and run Transit Least Squares. Returns (time, flux, results) or (None, None, None)."""
    url = BASE_URL + uri
    try:
        with fits.open(url) as hdu:
            time = hdu[1].data['TIME']
            flux = hdu[1].data['PDCSAP_FLUX']
    except Exception as e:
        print(f"  X Download failed: {e}")
        return None, None, None

    time, flux = cleaned_array(time, flux)

    if len(time) < 100:
        print(f"  X Too few points ({len(time)}), skipping.")
        return None, None, None

    flux = flux / numpy.median(flux)

    model   = transitleastsquares(time, flux)
    results = model.power(u=ab, period_min=0.5, period_max=13.5, oversampling_factor=3)
    return time, flux, results


def plot_sector(tic_id, sector_label, time, flux, results):
    """Generate the 4-panel plot for one sector and save it."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle(
        f"TIC {tic_id} — {sector_label}  |  SDE={results.SDE:.1f} P={results.period:.4f} d",
        fontsize=11, fontweight='bold'
    )

    # Panel 1: raw light curve
    axes[0, 0].scatter(time, flux, color='black', s=0.3, alpha=0.5, rasterized=True)
    axes[0, 0].set_xlabel('Time (BTJD)')
    axes[0, 0].set_ylabel('Normalized flux')
    axes[0, 0].set_title('Light curve')

    # Panel 2: TLS periodogram
    axes[0, 1].plot(results.periods, results.power, color='black', lw=0.5)
    axes[0, 1].axvline(results.period, alpha=0.5, lw=2, color='red',
                       label=f"P={results.period:.4f} d")
    for n in range(2, 6):
        axes[0, 1].axvline(n * results.period, alpha=0.15, lw=1, color='red', linestyle='--')
        axes[0, 1].axvline(results.period / n, alpha=0.15, lw=1, color='red', linestyle='--')
    axes[0, 1].set_xlim(0, max(results.periods))
    axes[0, 1].set_xlabel('Period (days)')
    axes[0, 1].set_ylabel('SDE')
    axes[0, 1].set_title('TLS Periodogram')
    axes[0, 1].legend(fontsize=8)

    # Panel 3: phase-folded transit
    half_dur = (results.duration / results.period) * 4
    axes[1, 0].plot(results.model_folded_phase, results.model_folded_model,
                    color='red', lw=1.5, zorder=3)
    axes[1, 0].scatter(results.folded_phase, results.folded_y,
                       color='blue', s=8, alpha=0.4, zorder=2)
    axes[1, 0].set_xlim(0.5 - half_dur, 0.5 + half_dur)
    axes[1, 0].set_xlabel('Phase')
    axes[1, 0].set_ylabel('Relative flux')
    axes[1, 0].set_title('Phase-folded transit')

    # Panel 4: per-transit depths
    axes[1, 1].errorbar(
        results.transit_times, results.transit_depths,
        yerr=results.transit_depths_uncertainties,
        fmt='o', color='red', markersize=4, label='Per-transit depth'
    )
    axes[1, 1].axhline(numpy.mean(results.transit_depths), color='black',
                       linestyle='dashed', label='Mean depth')
    axes[1, 1].axhline(1, color='gray', alpha=0.5)
    axes[1, 1].set_xlabel('Time (days)')
    axes[1, 1].set_ylabel('Flux')
    axes[1, 1].set_title('Individual transit depths')
    axes[1, 1].legend(fontsize=8)

    plt.tight_layout()
    safe_sector = sector_label.replace(' ', '_').replace('/', '-')
    plot_path = os.path.join(LOG_DIR, f"TIC_{tic_id}_{safe_sector}.png")
    plt.savefig(plot_path, bbox_inches='tight')
    plt.close()
    return plot_path

In [ ]:
# — Adjust these before hitting Run All —————————————————————————
MAX_STARS = 100      # number of stars to process; set to None for full run
TMAG_MIN  = 8.0
TMAG_MAX  = 13.0     # skip stars fainter than this (TLS unreliable beyond ~13)
SPOC_ONLY = True     # True = SPOC pipeline only (best quality); False = all authors
# ————————————————————————————————————————————————————————————————

# Upload previous log CSV if resuming
print("Upload results_log.csv if resuming (click Cancel/skip if starting fresh):")
uploaded_log = files.upload()
if uploaded_log:
    log_filename = list(uploaded_log.keys())[0]
    import shutil
    shutil.copy(log_filename, LOG_CSV)
    print(f"✓ Log restored — {log_filename}")
else:
    print("  No log uploaded — starting fresh.")

# Resume: skip already-logged TICs so you can restart without reprocessing
processed_tics = set()
if os.path.exists(LOG_CSV):
    done_df = pd.read_csv(LOG_CSV, on_bad_lines='skip')
    processed_tics = set(done_df['TIC_ID'].astype(int))
    print(f"Resuming — {len(processed_tics)} TICs already logged.")

tic_list = targets_df['TICID'].tolist()
if MAX_STARS is not None:
    tic_list = tic_list[:MAX_STARS]

print(f"Will process {len(tic_list)} targets  |  Log → {LOG_DIR}/")

Upload results_log.csv if resuming (click Cancel/skip if starting fresh):


Saving results_log (8).csv to results_log (8).csv
✓ Log restored — results_log (8).csv
Resuming — 67 TICs already logged.
Will process 100 targets  |  Log → tess_results/


In [ ]:
for idx, tic_id in enumerate(tic_list):
    print(f"\n{'-'*55}")
    print(f"  [{idx+1}/{len(tic_list)}]  TIC {tic_id}")
    print(f"{'-'*55}")

    # Already done?
    if tic_id in processed_tics:
        print("  → Already processed, skipping.")
        continue

    row_info = targets_df[targets_df['TICID'] == tic_id].iloc[0]
    tmag = row_info['Tmag']

    # Brightness filter (8.0 to 13.0)
    if pd.notna(tmag) and (float(tmag) > TMAG_MAX or float(tmag) < TMAG_MIN):
        skip_reason = 'SKIPPED_FAINT' if float(tmag) > TMAG_MAX else 'SKIPPED_BRIGHT'
        print(f"  → Outside brightness range (Tmag={tmag:.1f}), skipping.")
        append_log({'TIC_ID': tic_id, 'Tmag': tmag, 'RA': row_info['RA'],
                    'Dec': row_info['Dec'], 'Status': skip_reason,
                    'TOI_Disposition': '', 'Sectors_Found': 0,
                    'Best_Period_days': '', 'Best_SDE': '', 'Best_SNR': '',
                    'Transit_Depth': '', 'Transit_Duration_h': '',
                    'Distinct_Transits': '', 'Empty_Transits': '',
                    'Timestamp': datetime.utcnow().isoformat()})
        processed_tics.add(tic_id)
        continue

    # TOI catalog check
    should_skip, toi_reason, toi_disps = check_toi_status(tic_id)
    print(f"  TOI check: {toi_reason}")
    if should_skip:
        append_log({'TIC_ID': tic_id, 'Tmag': tmag, 'RA': row_info['RA'],
                    'Dec': row_info['Dec'], 'Status': 'SKIPPED_TOI',
                    'TOI_Disposition': ', '.join(str(d) for d in toi_disps),
                    'Sectors_Found': '', 'Best_Period_days': '', 'Best_SDE': '',
                    'Best_SNR': '', 'Transit_Depth': '', 'Transit_Duration_h': '',
                    'Distinct_Transits': '', 'Empty_Transits': '',
                    'Timestamp': datetime.utcnow().isoformat()})
        processed_tics.add(tic_id)
        continue

    # MAST search
    try:
        author = "SPOC" if SPOC_ONLY else None
        search_result = lk.search_lightcurve(f"TIC {tic_id}", author=author, mission="TESS")
    except Exception as e:
        print(f"  X MAST search failed: {e}")
        continue

    if len(search_result) == 0:
        print("  → No TESS light curves found.")
        append_log({'TIC_ID': tic_id, 'Tmag': tmag, 'RA': row_info['RA'],
                    'Dec': row_info['Dec'], 'Status': 'NO_DATA',
                    'TOI_Disposition': ', '.join(str(d) for d in toi_disps),
                    'Sectors_Found': 0, 'Best_Period_days': '', 'Best_SDE': '',
                    'Best_SNR': '', 'Transit_Depth': '', 'Transit_Duration_h': '',
                    'Distinct_Transits': '', 'Empty_Transits': '',
                    'Timestamp': datetime.utcnow().isoformat()})
        processed_tics.add(tic_id)
        continue

    n_sectors = len(search_result)
    print(f"  Found {n_sectors} sector(s)")

    # Limb darkening
    try:
        ab, mass, mass_min, mass_max, radius, radius_min, radius_max = catalog_info(TIC_ID=int(tic_id))
        print(f"  LD (a,b)={ab}  |  R*={radius:.2f} RO  M*={mass:.2f} MO")
    except Exception as e:
        print(f"  X catalog_info failed ({e}), using default LD.")
        ab = (0.4, 0.3)

    # Process every sector, keep best SDE
    best_result = None
    best_sector = None
    best_time   = None
    best_flux   = None

    for i, uri in enumerate(search_result.table['dataURL']):
        sector_label = search_result.table['mission'][i]
        print(f"\n  Sector {i+1}/{n_sectors}: {sector_label}")

        time_s, flux_s, results_s = run_tls_on_sector(tic_id, uri, sector_label, ab)

        if results_s is None:
            continue

         # NaN guard — catch both NaN SDE and NaN period
        import math
        if math.isnan(results_s.SDE) or math.isnan(results_s.period):
            print(f"  X TLS returned no valid fit (SDE={results_s.SDE:.2f}), skipping sector.")
            continue

        print(f"  SDE={results_s.SDE:.2f} P={results_s.period:.5f} d "
              f"Depth={results_s.depth:.5f} SNR={results_s.snr:.1f}")

        plot_sector(tic_id, sector_label, time_s, flux_s, results_s)

        if best_result is None or results_s.SDE > best_result.SDE:
            best_result = results_s
            best_sector = sector_label
            best_time   = time_s
            best_flux   = flux_s

    if best_result is None:
        print("  → All sectors failed to process.")
        processed_tics.add(tic_id)
        continue

    # Flag based on SDE threshold (standard: 7.1)
    sde_flag = 'CANDIDATE' if best_result.SDE >= 7.1 else \
               'WEAK'      if best_result.SDE >= 5.0 else 'NOISE'

    append_log({
        'TIC_ID':             tic_id,
        'Tmag':               tmag,
        'RA':                 row_info['RA'],
        'Dec':                row_info['Dec'],
        'Status':             sde_flag,
        'TOI_Disposition':    ', '.join(str(d) for d in toi_disps) if toi_disps else 'Not in TOI',
        'Sectors_Found':      n_sectors,
        'Best_Sector':        best_sector,
        'Best_Period_days':   round(best_result.period, 6),
        'Best_SDE':           round(best_result.SDE, 2),
        'Best_SNR':           round(best_result.snr, 2),
        'Transit_Depth':      round(best_result.depth, 6),
        'Transit_Duration_h': round(best_result.duration * 24, 3),
        'Distinct_Transits':  best_result.distinct_transit_count,
        'Empty_Transits':     best_result.empty_transit_count,
        'Timestamp':          datetime.utcnow().isoformat()
    })
    processed_tics.add(tic_id)

    symbol = '★' if sde_flag == 'CANDIDATE' else '·'
    print(f"\n  {symbol} Result: {sde_flag}  |  Best SDE={best_result.SDE:.2f}  P={best_result.period:.5f} d")

print(f"\n{'-'*55}")
print(f"  Done.  Logs saved to {LOG_CSV} and {LOG_XLSX}")
print(f"{'-'*55}")


-------------------------------------------------------
  [1/100]  TIC 1628071
-------------------------------------------------------
  → Already processed, skipping.

-------------------------------------------------------
  [2/100]  TIC 1650025
-------------------------------------------------------
  → Already processed, skipping.

-------------------------------------------------------
  [3/100]  TIC 5630042
-------------------------------------------------------
  → Already processed, skipping.

-------------------------------------------------------
  [4/100]  TIC 5647251
-------------------------------------------------------
  → Already processed, skipping.

-------------------------------------------------------
  [5/100]  TIC 5648410
-------------------------------------------------------
  → Already processed, skipping.

-------------------------------------------------------
  [6/100]  TIC 5649509
-------------------------------------------------------
  → Already process

100%|██████████| 2143/2143 periods | 00:47<00:00


Searching for best T0 for period 9.95949 days
  SDE=5.17 P=9.95949 d Depth=0.99973 SNR=16.2

  Sector 2/7: TESS Sector 36
Transit Least Squares TLS 1.32 (5 Apr 2024)
Creating model cache for 37 durations
Searching 15493 data points, 2127 periods from 0.601 to 11.948 days
Using all 2 CPU threads


100%|██████████| 2127/2127 periods | 00:44<00:00


Searching for best T0 for period 8.29360 days


...

100%|██████████| 2518/2518 periods | 00:47<00:00


Searching for best T0 for period 12.85262 days


/usr/local/lib/python3.12/dist-packages/transitleastsquares/main.py:411: UserWarning: 1 of 2 transits without data. The true period may be twice the given period.
  warnings.warn(text)


  SDE=4.70 P=12.85262 d Depth=0.98192 SNR=52.4

  ★ Result: CANDIDATE  |  Best SDE=9.61  P=4.41335 d

-------------------------------------------------------
  [100/100]  TIC 21109023
-------------------------------------------------------
  → Outside brightness range (Tmag=14.6), skipping.

-------------------------------------------------------
  Done.  Logs saved to tess_results/results_log.csv and tess_results/results_log.xlsx
-------------------------------------------------------


/tmp/ipykernel_627/4572142.py:130: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'Timestamp':          datetime.utcnow().isoformat()
/tmp/ipykernel_627/4572142.py:24: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'Timestamp': datetime.utcnow().isoformat()})


In [ ]:
# @title
# Validation — fetch TIC 261136679
TIC_ID = 261136679

ab, mass, mass_min, mass_max, radius, radius_min, radius_max = catalog_info(TIC_ID=TIC_ID)
print(f"LD (a,b)={ab}  |  R★={radius:.2f} R☉  M★={mass:.2f} M☉")

search_result = lk.search_lightcurve(f"TIC {TIC_ID}", author="SPOC", mission="TESS")
print(search_result)

In [ ]:
# @title
# Validation — run TLS and plot
best_result = None
best_sector = None
best_time   = None
best_flux   = None

for i, uri in enumerate(search_result.table['dataURL']):
    sector_label = search_result.table['mission'][i]
    print(f"\nSector {i+1}/{len(search_result)}: {sector_label}")

    time_s, flux_s, results_s = run_tls_on_sector(TIC_ID, uri, sector_label, ab)
    if results_s is None:
        continue
    if numpy.isnan(results_s.SDE) or numpy.isnan(results_s.period):
        print("    ✗ TLS returned NaN, skipping.")
        continue

    print(f"    SDE={results_s.SDE:.2f}  P={results_s.period:.5f} d  "
          f"Depth={results_s.depth:.5f}  SNR={results_s.snr:.1f}")

    # Plot with SDE threshold line added
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle(
        f"TIC {TIC_ID} — {sector_label}  |  SDE={results_s.SDE:.1f}  P={results_s.period:.4f} d",
        fontsize=11, fontweight='bold'
    )
    axes[0,0].scatter(time_s, flux_s, color='black', s=0.3, alpha=0.5, rasterized=True)
    axes[0,0].set_xlabel('Time (BTJD)')
    axes[0,0].set_ylabel('Normalized flux')
    axes[0,0].set_title('Light curve')

    axes[0,1].plot(results_s.periods, results_s.power, color='black', lw=0.5)
    axes[0,1].axvline(results_s.period, alpha=0.5, lw=2, color='red',
                      label=f"P={results_s.period:.4f} d")
    axes[0,1].axhline(7.1, color='blue', lw=1.2, linestyle='--', label='SDE threshold = 7.1')
    axes[0,1].set_xlim(0, max(results_s.periods))
    axes[0,1].set_xlabel('Period (days)')
    axes[0,1].set_ylabel('SDE')
    axes[0,1].set_title('TLS Periodogram')
    axes[0,1].legend(fontsize=8)

    half_dur = (results_s.duration / results_s.period) * 4
    axes[1,0].plot(results_s.model_folded_phase, results_s.model_folded_model,
                   color='red', lw=1.5, zorder=3)
    axes[1,0].scatter(results_s.folded_phase, results_s.folded_y,
                      color='blue', s=8, alpha=0.4, zorder=2)
    axes[1,0].set_xlim(0.5 - half_dur, 0.5 + half_dur)
    axes[1,0].set_xlabel('Phase')
    axes[1,0].set_ylabel('Relative flux')
    axes[1,0].set_title('Phase-folded transit')

    axes[1,1].errorbar(
        results_s.transit_times, results_s.transit_depths,
        yerr=results_s.transit_depths_uncertainties,
        fmt='o', color='red', markersize=4, label='Per-transit depth'
    )
    axes[1,1].axhline(numpy.mean(results_s.transit_depths), color='black',
                      linestyle='dashed', label='Mean depth')
    axes[1,1].axhline(1, color='gray', alpha=0.5)
    axes[1,1].set_xlabel('Time (days)')
    axes[1,1].set_ylabel('Flux')
    axes[1,1].set_title('Individual transit depths')
    axes[1,1].legend(fontsize=8)

    plt.tight_layout()
    save_path = f"/content/TIC_{TIC_ID}_{sector_label.replace(' ','_')}_validation.png"
    plt.savefig(save_path, bbox_inches='tight', dpi=500)
    plt.show()
    plt.close()
    print(f"    Saved: {save_path}")

    if best_result is None or results_s.SDE > best_result.SDE:
        best_result = results_s
        best_sector = sector_label
        best_time, best_flux = time_s, flux_s

if best_result is not None:
    print(f"\n{'━'*50}")
    print(f"  Best sector : {best_sector}")
    print(f"  SDE         : {best_result.SDE:.2f}")
    print(f"  Period      : {best_result.period:.5f} days")
    print(f"  Depth       : {best_result.depth:.6f}")
    print(f"  Duration    : {best_result.duration*24:.2f} hours")
    print(f"  Transits    : {best_result.distinct_transit_count}")
    print(f"{'━'*50}")